# ЛР 6. Исследование параметров text-to-image — вариант 18
Беспалый Максим Евгеньевич, Т26ИИКТ-МИК11. Бриф 3 «постер научной выставки» × фактор 6 «язык запроса: английский → русский».
Серия seed 101…105. Гипотеза, допуск и диагностические режимы C1–C3 записаны в журнал до запуска.

**Запуск:** «Среда выполнения» → «Сменить среду выполнения» → «Графический процессор T4» → «Сохранить» → «Выполнить все».

## Ячейка 1. Среда (шаг 1)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip list 2>/dev/null | grep -i -E "^(torch|diffusers|transformers|pillow|scipy|accelerate|safetensors) "
import sys, platform, datetime, torch
print("UTC:", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))
print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ GPU — смените среду выполнения на T4")

## Ячейка 2. lab05_lib.py (из ЛР 5, без изменений)

In [ ]:
%%writefile lab05_lib.py
# ЛР 5. Запуск открытой модели text-to-image. Библиотека учебных функций.
# Проверено 19.09.2026: Google Colab, GPU T4, Python 3.13, torch 2.11.0+cu128, diffusers 0.40.0, transformers 5.16.1.
# Модель: stabilityai/sd-turbo, ревизия закреплена (см. REVISION). Веса загружаются с Hugging Face без авторизации.
import hashlib, io, time
import numpy as np
import torch
from PIL import Image
from diffusers import AutoPipelineForText2Image
from diffusers.utils import logging as dlog

dlog.set_verbosity_error()  # убирает предупреждения о приведении типов при pipe.to(dtype)

MODEL_ID = "stabilityai/sd-turbo"
REVISION = "b261bac6fd2cf515557d5d0707481eafa0485ec2"

# Пять контекстов (брифов). Во всех запретены текст, логотипы и люди.
BRIEFS = [
    "editorial poster concept for a robotics festival, glowing mechanical arm silhouette, deep blue and amber palette, clean composition, no text, no logo, no people",
    "square podcast cover about the city environment, abstract layered skyline of geometric shapes, teal and orange palette, minimal flat style, no text, no people",
    "poster concept for a science exhibition, abstract constellation of connected nodes, dark background with white and cyan lines, balanced composition, no text, no logo",
    "visual for an ecology lecture series, stylized forest canopy seen from above, layered green shapes, soft morning light, no text, no people",
    "cover art for an electronic music release, brushed metal surface with iridescent reflections, macro photography style, strong contrast, no text, no logo, no people",
]


def load_pipe(dtype=torch.float16):
    """Загружает закреплённую ревизию (файлы fp16) на GPU; dtype задаёт тип вычислений."""
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID, revision=REVISION, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    pipe.set_progress_bar_config(disable=True)
    return pipe.to("cuda", dtype)


def generate(pipe, prompt, seed, steps=1, guidance=0.0, size=512, batch=1, gen_device="cpu"):
    """Возвращает список PIL-изображений. Для каждого элемента пакета свой генератор: seed, seed+1, ..."""
    gens = [torch.Generator(device=gen_device).manual_seed(seed + i) for i in range(batch)]
    return pipe(prompt=[prompt] * batch, num_inference_steps=steps, guidance_scale=guidance,
                height=size, width=size, generator=gens).images


def generate_fixed_noise(pipe, prompt, seed, steps=1, guidance=0.0, size=512):
    """Начальный шум всегда строится в float32 и приводится к типу вычислений: так меняется только точность."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    lat = torch.randn((1, 4, size // 8, size // 8), generator=g, dtype=torch.float32).to("cuda", pipe.unet.dtype)
    return pipe(prompt=[prompt], num_inference_steps=steps, guidance_scale=guidance, height=size, width=size, latents=lat).images[0]


def png_bytes(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def sha(data):
    return hashlib.sha256(data).hexdigest()


def to_arr(img):
    return np.asarray(img.convert("RGB"), dtype=np.float64)


def psnr_db(a, b):
    """PSNR для значений 0..255; inf, если изображения совпадают."""
    mse = np.mean((to_arr(a) - to_arr(b)) ** 2)
    return float("inf") if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)


def max_abs_diff(a, b):
    return int(np.max(np.abs(to_arr(a) - to_arr(b))))


def jpeg_roundtrip(img, quality=90):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB"), buf.getvalue()


def classify(psnr, tol_db=40.0):
    """Классификация пары по допуску, заданному до эксперимента."""
    return "identical" if psnr == float("inf") else ("close" if psnr >= tol_db else "different")


# Шесть условий сравнения (фактор варианта). Каждое условие даёт пару изображений A и B.
CONDITIONS = ["repeat", "seed_plus_1", "generator_device", "dtype", "file_format", "batch"]


def make_pair(pipe, prompt, seed, cond):
    if cond == "repeat":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed)[0]
    elif cond == "seed_plus_1":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed + 1)[0]
    elif cond == "generator_device":
        a = generate(pipe, prompt, seed, gen_device="cpu")[0]; b = generate(pipe, prompt, seed, gen_device="cuda")[0]
    elif cond == "dtype":
        pipe.to(torch.float16); a = generate(pipe, prompt, seed)[0]
        pipe.to(torch.float32); b = generate(pipe, prompt, seed)[0]
        pipe.to(torch.float16)
    elif cond == "file_format":
        a = generate(pipe, prompt, seed)[0]; b, _ = jpeg_roundtrip(a, 90)
    elif cond == "batch":
        a = generate(pipe, prompt, seed)[0]; b = generate(pipe, prompt, seed, batch=4)[0]
    else:
        raise ValueError(cond)
    return a, b


def variant_summary(pipe, n):
    """Вариант n = 1..30: контекст c = (n-1)//6 + 1, условие k = (n-1)%6 + 1."""
    c, k = (n - 1) // 6, (n - 1) % 6
    seed = 1000 + n
    t0 = time.perf_counter()
    a, b = make_pair(pipe, BRIEFS[c], seed, CONDITIONS[k])
    dt = time.perf_counter() - t0
    p = psnr_db(a, b)
    pf = None
    if CONDITIONS[k] == "dtype":  # проверка причины: одинаковый начальный шум в обоих режимах
        pipe.to(torch.float16); fa = generate_fixed_noise(pipe, BRIEFS[c], seed)
        pipe.to(torch.float32); fb = generate_fixed_noise(pipe, BRIEFS[c], seed)
        pipe.to(torch.float16); pf = psnr_db(fa, fb)
    return dict(variant=n, ctx=c + 1, cond=CONDITIONS[k], seed=seed, sha_a=sha(png_bytes(a)), sha_b=sha(png_bytes(b)),
                pixel_equal=bool(np.array_equal(to_arr(a), to_arr(b))), max_diff=max_abs_diff(a, b), psnr=p,
                verdict=classify(p), seconds=round(dt, 2), psnr_fixed_noise=pf)

## Ячейка 3. lab06_lib.py (приложение А МУ, без изменений)

In [ ]:
%%writefile lab06_lib.py
# ЛР 6. Исследование параметров text-to-image генерации. Библиотека учебных функций.
# Требуется файл lab05_lib.py из ЛР 5 в той же папке (из него берутся модель, генерация и брифы).
# Проверено 19.09.2026: Google Colab, GPU T4, Python 3.13, torch 2.11.0+cu128, diffusers 0.40.0, transformers 5.16.1.
import time
import numpy as np
import torch
from scipy import ndimage
from transformers import CLIPModel, CLIPProcessor
import lab05_lib as L

CLIP_ID = "openai/clip-vit-base-patch32"
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
SEEDS = range(101, 106)

# Короткие запросы (только предмет) и русские переводы брифов из ЛР 5.
SHORT = ["robotic arm poster", "podcast cover about the city", "science exhibition poster",
         "ecology lecture visual, forest", "electronic music cover art"]
RU = [
    "плакат-концепт для фестиваля робототехники, светящийся силуэт механической руки, глубокий синий и янтарный цвета, чистая композиция, без текста, без логотипа, без людей",
    "квадратная обложка подкаста о городской среде, абстрактный многослойный горизонт из геометрических фигур, бирюзовый и оранжевый цвета, минималистичный плоский стиль, без текста, без людей",
    "плакат-концепт научной выставки, абстрактное созвездие соединённых узлов, тёмный фон с белыми и голубыми линиями, сбалансированная композиция, без текста, без логотипа",
    "визуал для цикла лекций по экологии, стилизованный полог леса сверху, многослойные зелёные формы, мягкий утренний свет, без текста, без людей",
    "обложка электронного музыкального релиза, шлифованная металлическая поверхность с радужными бликами, стиль макросъёмки, сильный контраст, без текста, без логотипа, без людей",
]

# Шесть факторов: (имя, изменение A -> B). Базовый режим: структурированный английский запрос, 512, 1 шаг, guidance 0.
FACTORS = ["steps_1_to_4", "size_512_to_384", "size_512_to_768", "prompt_short_to_structured", "guidance_0_to_3", "language_en_to_ru"]


def load_clip():
    model = CLIPModel.from_pretrained(CLIP_ID, revision=CLIP_REVISION).to("cuda").eval()
    proc = CLIPProcessor.from_pretrained(CLIP_ID, revision=CLIP_REVISION)
    return model, proc


@torch.no_grad()
def clip_sim(clip, img, text):
    """Косинусное сходство изображения и текста в CLIP, умноженное на 100 (не CLIPScore)."""
    model, proc = clip
    inp = proc(text=[text], images=img, return_tensors="pt", padding=True, truncation=True).to("cuda")
    out = model(**inp)
    a = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    b = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((a * b).sum()) * 100


def luma(img):
    x = np.asarray(img.convert("RGB"), dtype=np.float64) / 255
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]


def edge_energy(img):
    """Средняя длина градиента яркости (оператор Собеля, нормировка 1/8), как в ЛР 4."""
    y = luma(img)
    return float(np.mean(np.hypot(ndimage.sobel(y, axis=0), ndimage.sobel(y, axis=1)) / 8))


def contrast(img):
    return float(np.std(luma(img)))


def settings(factor, c):
    """Пара режимов (A, B) для контекста c = 0..4: словарь параметров generate() плюс текст запроса."""
    base = dict(prompt=L.BRIEFS[c], steps=1, guidance=0.0, size=512)
    a, b = dict(base), dict(base)
    if factor == "steps_1_to_4":
        b["steps"] = 4
    elif factor == "size_512_to_384":
        b["size"] = 384
    elif factor == "size_512_to_768":
        b["size"] = 768
    elif factor == "prompt_short_to_structured":
        a["prompt"] = SHORT[c]
    elif factor == "guidance_0_to_3":
        b["guidance"] = 3.0
    elif factor == "language_en_to_ru":
        b["prompt"] = RU[c]
    else:
        raise ValueError(factor)
    return a, b


def measure(pipe, clip, cfg, c, seed):
    """Одно изображение: метрики и время генерации, с; сходство CLIP считается с исходным английским брифом."""
    t0 = time.perf_counter()
    img = L.generate(pipe, cfg["prompt"], seed, steps=cfg["steps"], guidance=cfg["guidance"], size=cfg["size"])[0]
    dt = time.perf_counter() - t0
    return dict(clip=clip_sim(clip, img, L.BRIEFS[c]), edge=edge_energy(img), contrast=contrast(img), seconds=dt), img


def variant_summary(pipe, clip, n, seeds=SEEDS):
    """Вариант n = 1..30: контекст c = (n-1)//6, фактор k = (n-1)%6. Для каждой метрики: A, s, B, разность, |d|/s, признак |d| > 2s."""
    c, k = (n - 1) // 6, (n - 1) % 6
    a_cfg, b_cfg = settings(FACTORS[k], c)
    A = {m: [] for m in ("clip", "edge", "contrast", "seconds")}
    B = {m: [] for m in A}
    for cfg in (a_cfg, b_cfg):  # прогрев: первый вызов с новым размером и числом шагов медленнее
        L.generate(pipe, cfg["prompt"], 0, steps=cfg["steps"], guidance=cfg["guidance"], size=cfg["size"])
    for s in seeds:
        ma, _ = measure(pipe, clip, a_cfg, c, s)
        mb, _ = measure(pipe, clip, b_cfg, c, s)
        for m in A:
            A[m].append(ma[m]); B[m].append(mb[m])
    rows = []
    for m in A:
        d = np.mean(B[m]) - np.mean(A[m]); sa = np.std(A[m], ddof=1)
        rows.append((m, float(np.mean(A[m])), float(sa), float(np.mean(B[m])), float(d), float(abs(d) / sa) if sa > 0 else float("inf"), bool(abs(d) > 2 * sa)))
    return dict(variant=n, ctx=c + 1, factor=FACTORS[k], rows=rows)

## Ячейка 4. Сверка библиотек с эталоном

In [ ]:
import hashlib, pathlib
for f, ref, size in (("lab05_lib.py", "4eb17650", 6447), ("lab06_lib.py", "c6d8473c", 6580)):
    b = pathlib.Path(f).read_bytes()
    for name, x in (("как записан", b), ("с \\n в конце", b.rstrip(b"\n") + b"\n")):
        h = hashlib.sha256(x).hexdigest()
        print(f"{f} {name}: {len(x)} байт, SHA-256 {h[:16]}…", "— совпадает с эталоном" if h.startswith(ref) else "")

## Ячейка 5. Загрузка модели и CLIP; режимы A и B (шаги 3–4)

In [ ]:
import json, time, importlib, numpy as np, torch, diffusers, transformers, PIL, scipy
import lab05_lib as L, lab06_lib as M
importlib.reload(L); importlib.reload(M)
N = 18
c, k = (N - 1) // 6, (N - 1) % 6
OUT = pathlib.Path("artifacts/v18"); OUT.mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter(); pipe = L.load_pipe(); t_pipe = time.perf_counter() - t0
t0 = time.perf_counter(); clip = M.load_clip(); t_clip = time.perf_counter() - t0
print(f"загрузка: SD Turbo {t_pipe:.1f} с, CLIP {t_clip:.1f} с")
a_cfg, b_cfg = M.settings(M.FACTORS[k], c)
print("фактор:", M.FACTORS[k], "| бриф", c + 1)
print("A:", a_cfg); print("B:", b_cfg)
print("различаются поля:", [key for key in a_cfg if a_cfg[key] != b_cfg[key]])
ENV = dict(device=torch.cuda.get_device_name(0), python=sys.version.split()[0], platform=platform.platform(),
           cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
           packages=dict(torch=torch.__version__, diffusers=diffusers.__version__, transformers=transformers.__version__,
                         pillow=PIL.__version__, scipy=scipy.__version__),
           sd_turbo=f"{L.MODEL_ID}@{L.REVISION}", clip=f"{M.CLIP_ID}@{M.CLIP_REVISION}", load_seconds=[round(t_pipe, 1), round(t_clip, 1)])

## Ячейка 6. Воспроизведение демонстрационного варианта 1 (продвинутый уровень)

In [ ]:
demo = M.variant_summary(pipe, clip, 1)
print("Вариант", demo["variant"], "фактор", demo["factor"], "(МУ: CLIP 30,796/1,394→30,292; границы 0,0457→0,0920; контраст 0,1923→0,2778)")
for m, a, s, b, d, ratio, sig in demo["rows"]:
    print(f"{m:9s} A={a:.4f} s={s:.4f} B={b:.4f} d={d:+.4f} |d|/s={ratio:.2f} 2s={'да' if sig else 'нет'}")

## Ячейка 7. Основной эксперимент: variant_summary(pipe, clip, 18) и классификация по допуску (шаги 5, 7)

In [ ]:
res = M.variant_summary(pipe, clip, N)
TOL = {"clip": ("abs", 1.0), "edge": ("rel", 0.10), "contrast": ("rel", 0.10), "seconds": ("rel", 0.20)}
def classify(m, a, d):
    kind, t = TOL[m]
    x = d if kind == "abs" else d / a
    if abs(x) <= t: return "нейтрально"
    if m == "seconds": return "ухудшение (медленнее)" if x > 0 else "улучшение (быстрее)"
    if m == "clip": return "улучшение" if x > 0 else "ухудшение"
    return "рост" if x > 0 else "падение"
print("Вариант", res["variant"], "фактор", res["factor"])
print(f"{'метрика':9s} {'A':>9s} {'s':>8s} {'B':>9s} {'Δ':>9s} {'Δ%':>8s} {'|Δ|/s':>7s} 2s   допуск")
ROWS = []
for m, a, s, b, d, ratio, sig in res["rows"]:
    cl = classify(m, a, d); ROWS.append(dict(metric=m, A=a, s=s, B=b, d=d, rel=d / a, ratio=ratio, sig2s=sig, tolerance=cl))
    print(f"{m:9s} {a:9.4f} {s:8.4f} {b:9.4f} {d:+9.4f} {100 * d / a:+7.1f}% {ratio:7.2f} {'да ' if sig else 'нет'}  {cl}")
print("\nКритерий 2s и допуск расходятся:", [r["metric"] for r in ROWS if r["sig2s"] != (r["tolerance"] != "нейтрально")])

# Значения по каждому seed (для отчёта и проверки разброса); изображения сохраняются
PER = {"A": [], "B": []}; IMGS = {"A": {}, "B": {}}
for s in M.SEEDS:
    for mode, cfg in (("A", a_cfg), ("B", b_cfg)):
        mm, img = M.measure(pipe, clip, cfg, c, s); PER[mode].append(dict(seed=s, **mm)); IMGS[mode][s] = img
        img.save(OUT / f"{mode}_seed{s}.png")
for s, ra, rb in zip(M.SEEDS, PER["A"], PER["B"]):
    print(f"seed {s}: CLIP {ra['clip']:.2f}→{rb['clip']:.2f}  границы {ra['edge']:.4f}→{rb['edge']:.4f}  контраст {ra['contrast']:.4f}→{rb['contrast']:.4f}")
print("CLIP (повторное измерение) совпадает с variant_summary:",
      np.isclose(np.mean([r["clip"] for r in PER["A"]]), ROWS[0]["A"]), np.isclose(np.mean([r["clip"] for r in PER["B"]]), ROWS[0]["B"]))

## Ячейка 8. Сравнительный лист comparison.png (шаг 6) и лист всей серии

In [ ]:
from PIL import Image, ImageDraw
pair = [L.generate(pipe, cfg["prompt"], 101, steps=cfg["steps"], guidance=cfg["guidance"], size=cfg["size"])[0].resize((384, 384)) for cfg in (a_cfg, b_cfg)]
sheet = Image.new("RGB", (768, 384)); sheet.paste(pair[0], (0, 0)); sheet.paste(pair[1], (384, 0))
sheet.save(OUT / "comparison.png"); display(sheet)
full = Image.new("RGB", (5 * 256, 2 * 256 + 24), "white"); dr = ImageDraw.Draw(full)
for j, s in enumerate(M.SEEDS):
    full.paste(IMGS["A"][s].resize((256, 256)), (j * 256, 24)); full.paste(IMGS["B"][s].resize((256, 256)), (j * 256, 280))
    dr.text((j * 256 + 4, 4), f"seed {s}: A (EN) сверху, B (RU) снизу", fill="black")
full.save(OUT / "comparison_series.png"); display(full.resize((5 * 160, 2 * 160 + 15)))

## Ячейка 9. Диагностика причины: длина запроса в токенах и режимы C1–C3
Режимы заданы в журнале до запуска. CLIP всегда считается относительно полного английского брифа.

In [ ]:
MODES = {
    "A_EN": L.BRIEFS[c],
    "B_RU": M.RU[c],
    "C1_EN_trunc": "poster concept for a science exhibition, abstract constellation of connected nodes",
    "C2_RU_compact": "постер выставки, созвездие узлов, тёмный фон, голубые линии, без текста",
    "C3_EN_compact": "exhibition poster, constellation of nodes, dark background, cyan lines, no text",
}
tok = pipe.tokenizer
TOK = {}
for name, p in MODES.items():
    n = len(tok(p).input_ids); TOK[name] = n
    kept = tok.decode(tok(p, truncation=True, max_length=77).input_ids[1:-1])
    print(f"{name:14s} токенов {n:3d}{' — УСЕЧЁН до 77' if n > 77 else ''}" + (f"\n{'':14s} модель видит: «{kept}»" if n > 77 else ""))
DIAG = {}
for name, p in MODES.items():
    vals = {"clip": [], "edge": [], "contrast": []}
    for s in M.SEEDS:
        mm, img = M.measure(pipe, clip, dict(prompt=p, steps=1, guidance=0.0, size=512), c, s)
        for key in vals: vals[key].append(mm[key])
        if s == 101: img.save(OUT / f"{name}_seed101.png")
    DIAG[name] = {key: (float(np.mean(v)), float(np.std(v, ddof=1))) for key, v in vals.items()}
    print(f"{name:14s} CLIP {DIAG[name]['clip'][0]:6.2f} ± {DIAG[name]['clip'][1]:.2f} | границы {DIAG[name]['edge'][0]:.4f} | контраст {DIAG[name]['contrast'][0]:.4f}")
print("\nэффект только усечения (C1 − A), CLIP:", round(DIAG["C1_EN_trunc"]["clip"][0] - DIAG["A_EN"]["clip"][0], 2))
print("эффект только языка (C2 − C3), CLIP:", round(DIAG["C2_RU_compact"]["clip"][0] - DIAG["C3_EN_compact"]["clip"][0], 2))
print("полный эффект (B − A), CLIP:", round(DIAG["B_RU"]["clip"][0] - DIAG["A_EN"]["clip"][0], 2))

# Справочно: как сама метрика CLIP ViT-B/32 относится к русскому тексту (не заменяет эталон!)
@torch.no_grad()
def text_emb(t):
    mdl, proc = clip; x = proc(text=[t], return_tensors="pt", padding=True, truncation=True).to("cuda")
    e = mdl.get_text_features(**x)
    if not torch.is_tensor(e):  # в transformers 5 может вернуться объект вывода модели
        e = e.text_embeds if getattr(e, "text_embeds", None) is not None else e.pooler_output
    return e / e.norm(dim=-1, keepdim=True)
TT = float((text_emb(L.BRIEFS[c]) * text_emb(M.RU[c])).sum()) * 100
RUSIM = float(np.mean([M.clip_sim(clip, IMGS["B"][s], M.RU[c]) for s in M.SEEDS]))
print(f"\nсправочно: сходство текстов EN и RU брифа в CLIP ×100 = {TT:.1f}; сходство изображений B с русским текстом = {RUSIM:.2f}")
strip = Image.new("RGB", (5 * 256, 256))
for j, name in enumerate(MODES): strip.paste(Image.open(OUT / f"{name}_seed101.png").resize((256, 256)), (j * 256, 0))
strip.save(OUT / "modes_seed101.png"); display(strip.resize((5 * 160, 160)))
print("слева направо:", list(MODES))

## Ячейка 10. Намеренная ошибка на своём брифе: guidance_scale = 1,0 (шаг 8)

In [ ]:
p = L.BRIEFS[c]
x0 = L.generate(pipe, p, 101, guidance=0.0)[0]
x1 = L.generate(pipe, p, 101, guidance=1.0)[0]; cfg1 = pipe.do_classifier_free_guidance
x3 = L.generate(pipe, p, 101, guidance=3.0)[0]; cfg3 = pipe.do_classifier_free_guidance
ERR = dict(sha_g0=L.sha(L.png_bytes(x0)), sha_g1=L.sha(L.png_bytes(x1)), g1_equals_g0=L.sha(L.png_bytes(x0)) == L.sha(L.png_bytes(x1)),
           cfg_active_g1=cfg1, cfg_active_g3=cfg3, psnr_g3_vs_g0=round(L.psnr_db(x0, x3), 2))
x3.save(OUT / "error_guidance3_seed101.png")
print(json.dumps(ERR, ensure_ascii=False, indent=2))

## Ячейка 11. Сводка и скачивание

In [ ]:
import uuid, shutil
SESSION = uuid.uuid4().hex[:8]
SUMMARY = dict(session=SESSION, utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), env=ENV, variant=N, factor=M.FACTORS[k],
               settings=dict(A=a_cfg, B=b_cfg), rows=ROWS, per_seed=PER, demo_variant1=demo["rows"], tokens=TOK,
               diagnostics=DIAG, text_text_clip=TT, clip_B_vs_ru_text=RUSIM, intentional_error=ERR)
(OUT / f"summary_{SESSION}.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
shutil.make_archive(f"lab06_v18_artifacts_{SESSION}", "zip", "artifacts")
print("Сеанс", SESSION)
try:
    from google.colab import files; files.download(f"lab06_v18_artifacts_{SESSION}.zip")
except Exception as e: print("Скачайте zip из панели «Файлы»:", e)

## После выполнения
«Файл» → «Скачать» → «.ipynb» (блокнот с выводом). По желанию: «Отключиться и удалить среду выполнения» → «Выполнить все»
ещё раз и сравнить таблицу ячейки 7 (CLIP, границы, контраст должны совпасть; время будет другим).